Homework 8 - Network Compression
===


## **Intro**

HW13 is about network compression

There are many types of Network/Model Compression,  here we introduce two:
* Knowledge Distillation 知識蒸餾
* Design Architecture 設計架構


The process of this notebook is as follows: <br/>
1. Introduce depthwise, pointwise and group convolution in MobileNet. 深度卷積、逐點卷積和分組卷積
2. Design the model of this colab
3. Introduce Knowledge-Distillation
4. Set up TeacherNet and it would be helpful in training


## **About the Dataset**  *(same as HW3)*

The dataset used here is food-11, a collection of food images in 11 classes.
本課程使用的資料集為 food-11，包含 11 類食物圖像。

For the requirement in the homework, TAs slightly modified the data.
Please DO NOT access the original fully-labeled training data or testing labels.
為了滿足作業要求，助教對資料集進行了少量修改。請勿存取原始的完整標註訓練資料或測試標籤。

Also, the modified dataset is for this course only, and any further distribution or commercial use is forbidden.

In [ ]:
### This block is same as HW3 ###
# Download the dataset
# You may choose where to download the data.

# Google Drive
!gdown --id '149paISvxCXDlr-720UEzseQA-y53rZxN' --output food-11.zip
# If you cannot successfully gdown, you can change a link. (Backup link is provided at the bottom of this colab tutorial).

# Dropbox
# !wget https://www.dropbox.com/s/m9q6273jl3djall/food-11.zip -O food-11.zip

# MEGA
# !sudo apt install megatools
# !megadl "https://mega.nz/#!zt1TTIhK!ZuMbg5ZjGWzWX1I6nEUbfjMZgCmAgeqJlwDkqdIryfg"

# Unzip the dataset.
# This may take some time.
!unzip -q food-11.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=149paISvxCXDlr-720UEzseQA-y53rZxN
From (redirected): https://drive.google.com/uc?id=149paISvxCXDlr-720UEzseQA-y53rZxN&confirm=t&uuid=5dc269a2-94bc-4f8f-85c0-2659762afc86
To: /content/food-11.zip
100% 958M/958M [00:13<00:00, 72.5MB/s]


## **Import Packages**  *(same as HW3)*

First, we need to import packages that will be used later.

In this homework, we highly rely on **torchvision**, a library of PyTorch.

In [ ]:
### This block is same as HW3 ###
# Import necessary packages.
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models

from PIL import Image

# "ConcatDataset" and "Subset" are possibly useful when doing semi-supervised learning.
from torch.utils.data import ConcatDataset, DataLoader, Subset
from torchvision.datasets import DatasetFolder

# This is for the progress bar.
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## **Dataset, Data Loader, and Transforms** *(similar to HW3)*

Torchvision provides lots of useful utilities for image preprocessing, data wrapping as well as data augmentation.
Torchvision 提供了許多影像預處理、資料封裝和資料增強的實用工具。

Here, since our data are stored in folders by class labels, we can directly apply **torchvision.datasets.DatasetFolder** for wrapping data without much effort.
由於我們的資料按類別標籤儲存在資料夾中，因此我們可以直接使用 `torchvision.datasets.DatasetFolder` 來進行資料封裝，無需太多額外操作。

Please refer to [PyTorch official website](https://pytorch.org/vision/stable/transforms.html) for details about different transforms.

---
**The only diffference with HW3 is that the transform functions are different.**

In [ ]:
### This block is similar to HW3 ###
# It is important to do data augmentation in training.
# However, not every augmentation is useful.
# Please think about what kind of augmentation is helpful for food recognition.

train_tfm = transforms.Compose(
    [
        transforms.Resize((142, 142)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.RandomCrop(128),
        transforms.ToTensor(),
    ]
)

test_tfm = transforms.Compose(
    [
        transforms.Resize((142, 142)),
        transforms.CenterCrop(128),
        transforms.ToTensor(),
    ]
)

In [ ]:
### This block is similar to HW3 ###
# Batch size for training, validation, and testing.
# A greater batch size usually gives a more stable gradient.
# But the GPU memory is limited, so please adjust it carefully.
batch_size = 64

# Construct datasets.
# The argument "loader" tells how torchvision reads the data.
train_set = DatasetFolder(
    "food-11/training/labeled",
    loader=lambda x: Image.open(x),
    extensions="jpg",
    transform=train_tfm,
)
valid_set = DatasetFolder(
    "food-11/validation",
    loader=lambda x: Image.open(x),
    extensions="jpg",
    transform=test_tfm,
)
unlabeled_set = DatasetFolder(
    "food-11/training/unlabeled",
    loader=lambda x: Image.open(x),
    extensions="jpg",
    transform=train_tfm,
)
test_set = DatasetFolder(
    "food-11/testing",
    loader=lambda x: Image.open(x),
    extensions="jpg",
    transform=test_tfm,
)

# Construct data loaders.
train_loader = DataLoader(
    train_set, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True
)
valid_loader = DataLoader(
    valid_set, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True
)
test_loader = DataLoader(
    test_set, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True
)

# **Architecture / Model Design**
The following are types of convolution layer design that has fewer parameters.以下是一些參數較少的捲積層設計類型。

## **Depthwise & Pointwise Convolution**
![](https://i.imgur.com/FBgcA0s.png)
> Blue: the connection between layers \
> Green: the expansion of **receptive field** \
> (reference: arxiv:1810.04231)

(a) normal convolution layer: It is fully connected. The difference between fully connected layer and fully connected convolution layer is the operation. (multiply --> convolution)

(b) Depthwise convolution layer(DW): You can consider each feature map pass through their own filter and then pass through pointwise convolution layer(PW) to combine the information of all pixels in feature maps.


(c) Group convolution layer(GC): Group the feature maps. Each group passes their filter then concate together. If group_size = input_feature_size, then GC becomes DC (channels are independent). If group_size = 1, then GC becomes fully connected.

<img src="https://i.imgur.com/Hqhg0Q9.png" width="500px">


## **Implementation details**
```python
# Regular Convolution, # of params = in_chs * out_chs * kernel_size^2
nn.Conv2d(in_chs, out_chs, kernel_size, stride, padding)

# Group Convolution, "groups" controls the connections between inputs and
# outputs. in_chs and out_chs must both be divisible by groups.
nn.Conv2d(in_chs, out_chs, kernel_size, stride, padding, groups=groups)

# Depthwise Convolution, out_chs=in_chs=groups, # of params = in_chs * kernel_size^2
nn.Conv2d(in_chs, out_chs=in_chs, kernel_size, stride, padding, groups=in_chs)

# Pointwise Convolution, a.k.a 1 by 1 convolution, # of params = in_chs * out_chs
nn.Conv2d(in_chs, out_chs, 1)

# Merge Depthwise and Pointwise Convolution (without )
def dwpw_conv(in_chs, out_chs, kernel_size, stride, padding):
    return nn.Sequential(
        nn.Conv2d(in_chs, in_chs, kernels, stride, padding, groups=in_chs),
        nn.Conv2d(in_chs, out_chs, 1),
    )
```

## **Model**

The basic model here is simply a stack of convolutional layers followed by some fully-connected layers. You can take advatage of depthwise & pointwise convolution to make your model deeper, but still follow the size constraint. 這裡的基本模型就是一系列卷積層堆疊起來，後面跟著一些全連接層。你可以利用深度可分離卷積和逐點卷積來加深模型，但仍然要遵守尺寸限制。

In [ ]:
class StudentNet(nn.Module):
    def __init__(self):
        super(StudentNet, self).__init__()

        def dwpw_conv(in_chs, out_chs, kernel_size=3, stride=1, padding=1):
            return nn.Sequential(
                nn.Conv2d(
                    in_chs,
                    in_chs,
                    kernel_size,
                    stride,
                    padding,
                    groups=in_chs,
                    bias=False,
                ),
                nn.BatchNorm2d(in_chs),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_chs, out_chs, 1, bias=False),
                nn.BatchNorm2d(out_chs),
                nn.ReLU(inplace=True),
            )

        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            dwpw_conv(32, 32, stride=2),  # 64x64
            dwpw_conv(32, 64, stride=1),
            dwpw_conv(64, 64, stride=2),  # 32x32
            dwpw_conv(64, 128, stride=1),
            dwpw_conv(128, 128, stride=2),  # 16x16
            dwpw_conv(128, 256, stride=1),
            dwpw_conv(256, 256, stride=2),  # 8x8
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(256, 11)

    def forward(self, x):
        out = self.cnn(x)
        out = out.view(out.size(0), -1)
        return self.fc(out)

## **Model Analysis**

Use `torchsummary` to get your model architecture (screenshot or pasting text are allowed.) and numbers of
parameters, these two information should be submit to your NTU Cool questions.
使用 torchsummary 取得模型架構（允許截圖或貼上文字）和參數數量，並將這兩個資訊提交到你的 NTU Cool 問題中。


Note that the number of parameters **should not greater than 100,000**, or you'll get penalty in this homework.
請注意，參數數量不得超過 100,000，否則本次作業將被扣分。

In [ ]:
from torchsummary import summary

student_net = StudentNet().to(device)
summary(student_net, (3, 128, 128), device=device)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 126, 126]             896
       BatchNorm2d-2         [-1, 32, 126, 126]              64
              ReLU-3         [-1, 32, 126, 126]               0
            Conv2d-4         [-1, 32, 124, 124]           9,248
       BatchNorm2d-5         [-1, 32, 124, 124]              64
              ReLU-6         [-1, 32, 124, 124]               0
         MaxPool2d-7           [-1, 32, 62, 62]               0
            Conv2d-8           [-1, 64, 60, 60]          18,496
       BatchNorm2d-9           [-1, 64, 60, 60]             128
             ReLU-10           [-1, 64, 60, 60]               0
        MaxPool2d-11           [-1, 64, 30, 30]               0
           Conv2d-12          [-1, 100, 28, 28]          57,700
      BatchNorm2d-13          [-1, 100, 28, 28]             200
             ReLU-14          [-1, 100,

## **Knowledge Distillation**

<img src="https://i.imgur.com/H2aF7Rv.png=100x" width="500px">

Since we have a learned big model, let it teach the other small model. In implementation, let the training target be the prediction of big model instead of the ground truth.
既然我們已經訓練了一個大型模型，就讓它來訓練另一個小型模型。在實現過程中，將大型模型的預測結果作為訓練目標，而不是使用真實值。

## **Why it works?**
* If the data is not clean, then the prediction of big model could ignore the noise of the data with wrong labeled. 如果資料不乾淨，大型模型的預測可能會忽略錯誤標籤造成的雜訊。
* The labels might have some relations. Number 8 is more similar to 6, 9, 0 than 1, 7, for example. 標籤之間可能存在某種關聯。例如，數字 8 與 6、9、0 的相似度高於與 1、7 的相似度。


## **How to implement?**
* $Loss = \alpha T^2 \times KL(\frac{\text{Teacher's Logits}}{T} || \frac{\text{Student's Logits}}{T}) + (1-\alpha)(\text{Original Loss})$
* Note that the logits here should have passed softmax.

In [ ]:
def loss_fn_kd(outputs, labels, teacher_outputs, T=6, alpha=0.7):
    """
    T: Temperature（建議 4~8）
    alpha: 軟標籤權重（建議 0.5~0.9）
    """
    hard_loss = F.cross_entropy(outputs, labels) * (1.0 - alpha)

    soft_loss = nn.KLDivLoss(reduction="batchmean")(
        F.log_softmax(outputs / T, dim=1), F.softmax(teacher_outputs / T, dim=1)
    ) * (alpha * T * T)

    return hard_loss + soft_loss

## **Teacher Model Setting**
We provide a well-trained teacher model to help you knowledge distillation to student model.
Note that if you want to change the transform function, you should consider  if suitable for this well-trained teacher model. 我們提供了一個訓練有素的教師模型，幫助您將知識蒸餾到學生模型。請注意，如果您想要更改轉換函數，則應考慮其是否適用於此訓練有素的教師模型。

* If you cannot successfully gdown, you can change a link. (Backup link is provided at the bottom of this colab tutorial). 如果您無法成功下載，可以嘗試使用備用連結。 （本 Colab 教程底部提供了備用連結。）


In [ ]:
# Download teacherNet
!gdown --id '1ni4IQUJ4bJJX4toHpvdXIu4VW3P29QrH' --output teacher_net.ckpt
# Load teacherNet
teacher_net = torch.load("./teacher_net.ckpt", weights_only=False)
teacher_net.eval()

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1ni4IQUJ4bJJX4toHpvdXIu4VW3P29QrH
From (redirected): https://drive.google.com/uc?id=1ni4IQUJ4bJJX4toHpvdXIu4VW3P29QrH&confirm=t&uuid=084f0bb8-9db1-4092-95df-87b87f47d818
To: /content/teacher_net.ckpt
100% 44.8M/44.8M [00:00<00:00, 126MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## **Generate Pseudo Labels in Unlabeled Data**

Since we have a well-trained model, we can use this model to predict pseudo-labels and help the student network train well. Note that you
**CANNOT** use well-trained model to pseudo-label the test data. 由於我們已經有一個訓練良好的模型，我們可以利用這個模型來預測偽標籤，從而幫助學生網路更好地訓練。請注意，您不能使用訓練良好的模型來偽標註測試資料。


---

**AGAIN, DO NOT USE TEST DATA FOR PURPOSE OTHER THAN INFERENCING**
再次強調，請勿將測驗資料用於推理以外的任何用途。

* Because If you use teacher network to predict pseudo-labels of the test data, you can only use student network to overfit these pseudo-labels without train/unlabeled data. In this way, your kaggle accuracy will be as high as the teacher network, but the fact is that you just overfit the test data and your true testing accuracy is very low.
* These contradict the purpose of these assignment (network compression); therefore, you should not misuse the test data.
* If you have any concerns, you can email us.


In [ ]:
# "cuda" only when GPUs are available.
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize a model, and put it on the device specified.
student_net = student_net.to(device)
teacher_net = teacher_net.to(device)

# Whether to do pseudo label.
do_semi = False


def get_pseudo_labels(dataset, model):
    loader = DataLoader(
        dataset, batch_size=batch_size * 3, shuffle=False, pin_memory=True
    )
    pseudo_labels = []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(loader):
            img, _ = batch
            logits = model(img.to(device))
            pseudo_labels.append(logits.argmax(dim=-1).detach().cpu())
    pseudo_labels = torch.cat(pseudo_labels)
    
    for idx, ((img, _), pseudo_label) in enumerate(zip(dataset.samples, pseudo_labels)):
        dataset.samples[idx] = (img, pseudo_label.item())
    return dataset


if do_semi:
    # Generate new trainloader with unlabeled set.
    unlabeled_set = get_pseudo_labels(unlabeled_set, teacher_net)
    concat_dataset = ConcatDataset([train_set, unlabeled_set])
    train_loader = DataLoader(
        concat_dataset,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=True,
        drop_last=True,
        num_workers=0,
    )

## **Training** *(similar to HW3)*

You can finish supervised learning by simply running the provided code without any modification. 您只需執行提供的程式碼即可完成監督學習，無需任何修改。

The function "get_pseudo_labels" is used for semi-supervised learning.
It is expected to get better performance if you use unlabeled data for semi-supervised learning.
However, you have to implement the function on your own and need to adjust several hyperparameters manually. 函數“get_pseudo_labels”用於半監督學習。使用未標記資料進行半監督學習有望獲得更好的效能。但是，您需要自行實作該函數，並手動調整一些超參數。

For more details about semi-supervised learning, please refer to [Prof. Lee's slides](https://speech.ee.ntu.edu.tw/~tlkagk/courses/ML_2016/Lecture/semi%20(v3).pdf).

Again, please notice that utilizing external data (or pre-trained model) for training is **prohibited**.
再次提醒，禁止使用外部資料（或預訓練模型）進行訓練。

---
**The only diffference with HW3 is that you should use loss in  knowledge distillation.**
與作業3的唯一區別在於，您需要在知識蒸餾中使用損失函數。




In [ ]:
# For the classification task, we use cross-entropy as the measurement of performance.
criterion = nn.CrossEntropyLoss()

# Initialize optimizer, you may fine-tune some hyperparameters such as learning rate on your own.
# Optimizer 建議改用較小 lr + weight decay
optimizer = torch.optim.AdamW(student_net.parameters(), lr=3e-4, weight_decay=1e-4)

# The number of training epochs.
n_epochs = 80

# 可加入 scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

for epoch in range(n_epochs):
    # ---------- Training ----------
    # Make sure the model is in train mode before training.
    student_net.train()

    # These are used to record information in training.
    train_loss, train_accs = [], []

    # Iterate the training set by batches.
    for batch in tqdm(train_loader):
        # A batch consists of image data and corresponding labels.
        imgs, labels = batch
        imgs, labels = imgs.to(device), labels.to(device)

        # Forward the data. (Make sure data and model are on the same device.)
        logits = student_net(imgs)
        # Teacher net will not be updated. And we use torch.no_grad
        # to tell torch do not retain the intermediate values
        # (which are for backpropgation) and save the memory.
        with torch.no_grad():
            soft_labels = teacher_net(imgs)

        # Calculate the loss in knowledge distillation method.
        loss = loss_fn_kd(logits, labels, soft_labels)

        # Gradients stored in the parameters in the previous step should be cleared out first.
        optimizer.zero_grad()

        # Compute the gradients for parameters.
        loss.backward()

        # Clip the gradient norms for stable training.
        # grad_norm = nn.utils.clip_grad_norm_(student_net.parameters(), max_norm=10)
        nn.utils.clip_grad_norm_(student_net.parameters(), max_norm=10)

        # Update the parameters with computed gradients.
        optimizer.step()

        # Compute the accuracy for current batch.
        acc = (logits.argmax(dim=-1) == labels).float().mean().item()

        # Record the loss and accuracy.
        train_loss.append(loss.item())
        train_accs.append(acc)

    # The average loss and accuracy of the training set is the average of the recorded values.
    train_loss_avg = sum(train_loss) / len(train_loss)
    train_acc_avg = sum(train_accs) / len(train_accs)

    # Print the information.
    print(
        f"[ Train | {epoch + 1:03d}/{n_epochs:03d} ] loss = {train_loss_avg:.5f}, acc = {train_acc_avg:.5f}"
    )

    # ---------- Validation ----------
    # Make sure the model is in eval mode so that some modules like dropout are disabled and work normally.
    student_net.eval()

    # These are used to record information in validation.
    valid_loss, valid_accs = [], []

    # Iterate the validation set by batches.
    with torch.no_grad():
        for batch in tqdm(valid_loader):
            # A batch consists of image data and corresponding labels.
            imgs, labels = batch
            imgs, labels = imgs.to(device), labels.to(device)

            # We don't need gradient in validation.
            # Using torch.no_grad() accelerates the forward process.
            logits = student_net(imgs)
            soft_labels = teacher_net(imgs)
            # We can still compute the loss (but not the gradient).
            loss = loss_fn_kd(logits, labels, soft_labels)

            # Compute the accuracy for current batch.
            acc = (logits.argmax(dim=-1) == labels).float().mean().item()

            # Record the loss and accuracy.
            valid_loss.append(loss.item())
            valid_accs.append(acc)

    # The average loss and accuracy for entire validation set is the average of the recorded values.
    valid_loss_avg = sum(valid_loss) / len(valid_loss)
    valid_acc_avg = sum(valid_accs) / len(valid_accs)

    # Print the information.
    print(
        f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss_avg:.5f}, acc = {valid_acc_avg:.5f}"
    )

    if scheduler:
        scheduler.step()

  0%|          | 0/49 [00:00<?, ?it/s]

[ Train | 001/080 ] loss = 1.23710, acc = 0.09120


  0%|          | 0/11 [00:00<?, ?it/s]

[ Valid | 001/080 ] loss = 1.21424, acc = 0.07879


  0%|          | 0/49 [00:00<?, ?it/s]

...

  0%|          | 0/49 [00:00<?, ?it/s]

[ Train | 013/080 ] loss = 1.23887, acc = 0.08801


  0%|          | 0/11 [00:00<?, ?it/s]

[ Valid | 013/080 ] loss = 1.22390, acc = 0.09091


  0%|          | 0/49 [00:00<?, ?it/s]

## **Testing** *(same as HW3)*

For inference, we need to make sure the model is in eval mode, and the order of the dataset should not be shuffled ("shuffle=False" in test_loader). 為了進行推理，我們需要確保模型處於評估模式，並且資料集的順序不能被打亂（在 test_loader 中設定「shuffle=False」）。

Last but not least, don't forget to save the predictions into a single CSV file.
The format of CSV file should follow the rules mentioned in the slides. 最後，別忘了將預測結果儲存到一個 CSV 檔案中。 CSV 檔案的格式應遵循幻燈片中提到的規則。

### **WARNING -- Keep in Mind**

Cheating includes but not limited to:
1.   using testing labels, 使用測試標籤；
2.   submitting results to previous Kaggle competitions,
3.   sharing predictions with others,
4.   copying codes from any creatures on Earth,
5.   asking other people to do it for you.

Any violations bring you punishments from getting a discount on the final grade to failing the course.

It is your responsibility to check whether your code violates the rules.
When citing codes from the Internet, you should know what these codes exactly do.
You will **NOT** be tolerated if you break the rule and claim you don't know what these codes do.


In [ ]:
### This block is same as HW3 ###
# Make sure the model is in eval mode.
# Some modules like Dropout or BatchNorm affect if the model is in training mode.
student_net.eval()

# Initialize a list to store the predictions.
predictions = []

# Iterate the testing set by batches.
for batch in tqdm(test_loader):
    # A batch consists of image data and corresponding labels.
    # But here the variable "labels" is useless since we do not have the ground-truth.
    # If printing out the labels, you will find that it is always 0.
    # This is because the wrapper (DatasetFolder) returns images and labels for each batch,
    # so we have to create fake labels to make it work normally.
    imgs, labels = batch

    # We don't need gradient in testing, and we don't even have labels to compute loss.
    # Using torch.no_grad() accelerates the forward process.
    with torch.no_grad():
        logits = student_net(imgs.to(device))

    # Take the class with greatest logit as prediction and record it.
    predictions.extend(logits.argmax(dim=-1).cpu().numpy().tolist())

In [ ]:
### This block is same as HW3 ###
# Save predictions into the file.
with open("predict.csv", "w") as f:
    # The first row must be "Id, Category"
    f.write("Id,Category\n")

    # For the rest of the rows, each image id corresponds to a predicted class.
    for i, pred in enumerate(predictions):
        f.write(f"{i},{pred}\n")